## 4. Agent Memory with a Context Provider

**Goal**: Remember a user's name in session state and inject it into later agent calls using `ContextProvider`.

Run the cells in order using the project's `.venv` kernel. Configure `AZURE_AI_PROJECT_ENDPOINT_FULL` and `AZURE_AI_MODEL_DEPLOYMENT_NAME` in `.env`, and authenticate with `az login`.

This is application-side, in-memory session state, not a Foundry memory store. Reusing the session preserves the name; creating a new session or restarting the kernel starts fresh.

In [1]:
# Copyright (c) Microsoft. All rights reserved.

import os
import sys
from typing import Any

sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from dotenv import load_dotenv
from agent_framework import Agent, AgentSession, ContextProvider, SessionContext
from agent_framework.foundry import FoundryChatClient

load_dotenv()

from utils.config import get_env, get_credential

PROJECT_ENDPOINT = get_env("AZURE_AI_PROJECT_ENDPOINT_FULL")
MODEL = get_env("AZURE_AI_MODEL_DEPLOYMENT_NAME")

client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL,
    credential=get_credential(),
)

c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


2026-10-26 11:26:07,537 - azure.ai.agentserver - INFO - Application Insights not configured, telemetry export disabled.


### Part A - Define the memory provider

`before_run` adds instructions using the saved name, or asks the agent to request it. `after_run` looks for "my name is" in the input and saves the first word as the name for subsequent calls.

This intentionally simple parser is for demonstration, not general-purpose name extraction. The framework manages the provider's state separately for each session.

In [2]:
class UserMemoryProvider(ContextProvider):
    def __init__(self) -> None:
        super().__init__("user_memory")

    async def before_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        user_name = state.get("user_name")
        instructions = (
            f"The user's name is {user_name}. Address them by name."
            if user_name
            else "Ask for the user's name."
        )
        context.extend_instructions(self.source_id, instructions)

    async def after_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        for message in context.input_messages:
            text = message.text
            if not isinstance(text, str):
                continue
            _, marker, name = text.lower().partition("my name is")
            if marker and name.strip():
                state["user_name"] = name.split()[0].capitalize()

### Part B - Create the agent and a shared session

All three turns below reuse this session. Rerun this cell to reset the conversation and the remembered name.

In [3]:
agent = Agent(
    client=client,
    name="UserMemoryAgent",
    instructions="You are a friendly assistant.",
    context_providers=[UserMemoryProvider()],
)
session = agent.create_session()

### Part C - Run a multi-turn conversation

On the first turn, the agent should ask for your name. The second turn saves **Alice** after the response. On the third turn, the provider injects the saved name so the agent can address Alice while answering the math question.

Jupyter supports top-level `await`, so no `asyncio.run()` or script entry point is needed. Exact model responses may vary.

In [4]:
print(await agent.run("Hello! What's the square root of 9?", session=session))
print(await agent.run("My name is Alice", session=session))
print(await agent.run("What is 2 + 2?", session=session))

Hello! The square root of 9 is 3. What's your name?
Nice to meet you, Alice! How can I assist you today?
2 + 2 equals 4. Anything else you'd like to know, Alice?


In [5]:
print(await agent.run("Hello! What's the square root of 9?", session=session))
print(await agent.run("What is 2 + 2?", session=session))

Hi again, Alice! The square root of 9 is still 3. Anything else you're curious about?
2 + 2 is 4. Need help with anything else, Alice?
